## NB02-Data transformation

### Load raw data

In [1]:
import json
import pandas as pd

In [2]:
with open("../data/raw/movies.json", "r") as f:
    all_pages = json.load(f)

with open("../data/raw/genres.json", "r") as f:
    genres_raw = json.load(f)

### Decision: flatten pages

In [3]:
all_movies = []
for page_data in all_pages:
    all_movies.extend(page_data["results"]) #`.extend()` adds all items from a list individually (unlike `.append()`, which would add the whole `results` list as one nested item). 

print(f"Total movie records collected: {len(all_movies)}")

Total movie records collected: 10000




`all_pages` is a list of 50 API responses — one dict per page, each holding
its own `results` list of 20 movies  To analyse movies as a single
table later, we need one flat list of movie records instead of 50 separate
page-dicts.



### Build the movies and genre DataFrames

In [4]:
genres_df = pd.DataFrame(genres_raw["genres"])
genres_df.head()

,id,name
0,28,Action
1,12,Adventure
2,16,Animation
3,35,Comedy
4,80,Crime


In [5]:
movies_df = pd.DataFrame(all_movies)
movies_df.head()

,adult,backdrop_path,genre_ids,id,title,original_language,original_title,overview,popularity,poster_path,release_date,softcore,video,vote_average,vote_count
0,False,/vjMvFSmGUxEtqVdaZgvFee9XkZl.jpg,"[878, 28, 12]",969681,Spider-Man: Brand New Day,en,Spider-Man: Brand New Day,Fighting crime full-time as Spider-Man in a wo...,1519.8510,/iPOn6DinuVyLY17YM9mKuPofV08.jpg,2026-07-28,False,False,8.000,301
1,False,/RMXG8myu1aGlNUsRjtxzmpdMK0.jpg,"[12, 28, 14]",1368337,The Odyssey,en,The Odyssey,"Odysseus, the legendary King of Ithaca, embark...",1166.5658,/5rhTDKUhPYvpdQIijFIs5VoWsON.jpg,2026-07-15,False,False,7.944,1859
2,False,/54KIfdTEzOliHDKx0OkzYGqAICx.jpg,"[28, 12, 878]",1081003,Supergirl,en,Supergirl,When an unexpected and ruthless adversary stri...,630.4075,/1QCWdqzTfh2x9UylVpspIU6QTuM.jpg,2026-06-24,False,False,6.800,1139
3,False,/piV2OnzTZCyGBP9JCjlHIgKGlfo.jpg,"[28, 14, 878]",454639,Masters of the Universe,en,Masters of the Universe,"After being separated for 15 years, the Sword ...",486.7027,/oRuyGUHdoaQxWP3SDfafGkStxTC.jpg,2026-06-03,False,False,7.285,1347
4,False,/flxau5Iu7bChQHsESqvGZ3FQRaI.jpg,"[878, 53]",1275779,Disclosure Day,en,Disclosure Day,A cybersecurity expert becomes a whistleblower...,397.2506,/AnJ8IQJI23hNpYXVNaythu061Ru.jpg,2026-06-10,False,False,7.407,2040


### Deleting duplicated values

In [6]:
print("Duplicate movie ids:", movies_df["id"].duplicated().sum())
movies_df = movies_df.drop_duplicates(subset="id")
print("After filtering:", len(movies_df))

Duplicate movie ids: 354
After filtering: 9646


### Filtering the data set

Analizing the `vote_count` and `popularity`. TMDB's `popularity` is a platform-calculated score reflecting current interest and activity (page views, recent votes, watchlist/favourite additions, release
recency) — not perceived quality.

In [7]:
print("Minimum vote_count:", movies_df["vote_count"].min(), "/  Mean vote_count:", movies_df["vote_count"].mean(), "/  Maximum vote_count:", movies_df["vote_count"].max())
print("Minimum popularity:", movies_df["popularity"].min(), "/  Mean popularity:", movies_df["popularity"].mean(), "/  Maximum popularity:", movies_df["popularity"].max())

Minimum vote_count: 0 /  Mean vote_count: 1709.954074227659 /  Maximum vote_count: 40513
Minimum popularity: 2.628 /  Mean popularity: 8.588594349989634 /  Maximum popularity: 1519.851


Decision: remove Zero-vote movies

2,054 of 9,683 movies (21.2%) have `vote_count == 0` — no one has rated them yet.
These are not low-quality data points, they simply have no rating at all, and
their `vote_average` of 0.0 is not a real score but TMDB's placeholder for "no
votes". Including them would corrupt any average I compute per genre: a genre
with many unrated new releases would show an artificially low `vote_average`
that reflects lack of votes, not lack of quality.

This is also why filtering on `vote_count` matters beyond reliability: it isn't
just trimming noisy low-vote films, it's removing a fifth of the dataset that
carries no genuine rating signal at all.

In [8]:
print("Before filtering:", len(movies_df))
print("Movies with no vote count:", (movies_df["vote_count"] == 0).sum())


Before filtering: 9646
Movies with no vote count: 2760


In [9]:
movies_df=movies_df[movies_df["vote_count"] != 0]

In [10]:
print("After filtering:", len(movies_df))

After filtering: 6886


Analizyng bool columns 

In [11]:
bool_columns = movies_df.select_dtypes(include="bool").columns # `.select_dtypes(include="x")` returns only the columns whose data type is x.
for col in bool_columns:
    print(f"--- {col} ---")
    print(movies_df[col].value_counts())
    print()

--- adult ---
adult
False    6886
Name: count, dtype: int64

--- softcore ---
softcore
False    6886
Name: count, dtype: int64

--- video ---
video
False    6886
Name: count, dtype: int64



Decision: drop boolean columns. `Adult` , `softcore` and `video` showed  no variation ( all False), so they add no useful information.

In [12]:
movies_df = movies_df.drop(columns=bool_columns)
movies_df.columns

Index(['backdrop_path', 'genre_ids', 'id', 'title', 'original_language',
       'original_title', 'overview', 'popularity', 'poster_path',
       'release_date', 'vote_average', 'vote_count'],
      dtype='object')

Decision: drop columns not needed for the research question. `backdrop_path`, `poster_path` are image URLs (not analysable data). `overview` is free text, not needed for a genre/rating/popularity comparison. `original_title` is redundant with `title` for this analysis.


In [13]:
movies_df = movies_df.drop(columns=["backdrop_path", "poster_path", "overview", "original_title"])
movies_df.columns

Index(['genre_ids', 'id', 'title', 'original_language', 'popularity',
       'release_date', 'vote_average', 'vote_count'],
      dtype='object')

Search for null rows

In [14]:
for name, df in [("movies_df", movies_df), ("genres_df", genres_df)]:
    print(f"--- {name} ---")
    print(df.isna().sum())  # counts nulls per column
    print()

--- movies_df ---
genre_ids            0
id                   0
title                0
original_language    0
popularity           0
release_date         0
vote_average         0
vote_count           0
dtype: int64

--- genres_df ---
id      0
name    0
dtype: int64



Decision: extract release year to compare "over the years". 

In [15]:
movies_df["release_date"] = pd.to_datetime(movies_df["release_date"])
movies_df["release_year"] = movies_df["release_date"].dt.year
movies_df.head()

,genre_ids,id,title,original_language,popularity,release_date,vote_average,vote_count,release_year
0,"[878, 28, 12]",969681,Spider-Man: Brand New Day,en,1519.8510,2026-07-28,8.000,301,2026.0
1,"[12, 28, 14]",1368337,The Odyssey,en,1166.5658,2026-07-15,7.944,1859,2026.0
2,"[28, 12, 878]",1081003,Supergirl,en,630.4075,2026-06-24,6.800,1139,2026.0
3,"[28, 14, 878]",454639,Masters of the Universe,en,486.7027,2026-06-03,7.285,1347,2026.0
4,"[878, 53]",1275779,Disclosure Day,en,397.2506,2026-06-10,7.407,2040,2026.0


Decision: deleting missing release dates

`release_date` arrives from TMDB as an empty string for movies with no
confirmed release date, not as a true null — so a plain `isna()` check on the
raw column shows zero missing values even though some are genuinely absent.
Converting to datetime turns those
empty strings into `NaT`, which is when the gap becomes visible.

**14 movies have no usable release date.** These are titles TMDB has catalogued
but not yet confirmed a release date for, which means they cannot be assigned a
`release_year` or `decade`. Since the genre-by-decade analysis in NB03 requires
both, I drop these 8 rather than imputing a year for them.

In [16]:
n_before = len(movies_df)
movies_df = movies_df.dropna(subset=["release_year"])
print(f"Dropped {n_before - len(movies_df)} movies with no usable release date")

Dropped 14 movies with no usable release date


Decision: add a column od decade, to simplify historical evolution

In [17]:
movies_df["decade"] = (movies_df["release_year"] // 10 * 10).astype(int)

### Decision: merge the tables genre and movies

`genre_ids` are just numbers, merging with the genre lookup table replaces
them with real genre names, which makes posible grouping them by genre names for future exploration. Aditionally, I change the name of the column `name` for `genre`

In [18]:
movies_exploded = movies_df.explode("genre_ids") #genre_ids is a list per movie, .explode() turns each list item into its own row
movies_with_genres = movies_exploded.merge(
    genres_df,
    left_on="genre_ids",
    right_on="id",
    how="left",
    suffixes=("", "_genre")
)
movies_with_genres = movies_with_genres.drop(columns=["id_genre"])
movies_with_genres.columns
movies_with_genres = movies_with_genres.rename(columns={"name": "genre"})

movies_with_genres.head()

,genre_ids,id,title,original_language,popularity,release_date,vote_average,vote_count,release_year,decade,genre
0,878,969681,Spider-Man: Brand New Day,en,1519.8510,2026-07-28,8.000,301,2026.0,2020,Science Fiction
1,28,969681,Spider-Man: Brand New Day,en,1519.8510,2026-07-28,8.000,301,2026.0,2020,Action
2,12,969681,Spider-Man: Brand New Day,en,1519.8510,2026-07-28,8.000,301,2026.0,2020,Adventure
3,12,1368337,The Odyssey,en,1166.5658,2026-07-15,7.944,1859,2026.0,2020,Adventure
4,28,1368337,The Odyssey,en,1166.5658,2026-07-15,7.944,1859,2026.0,2020,Action


### Save the prepared data 

In [19]:
movies_df.to_csv("../data/processed/movies.csv", index=False)
genres_df.to_csv("../data/processed/genres.csv", index=False)
movies_with_genres.to_csv("../data/processed/movies_with_genres.csv", index=False)